# OCR training — 5,000 total pairs split 80/10/10
Exactly 4,000 training, 500 validation, 500 held-out test line-image/text pairs, with repeated normalised text, original crop, known real photo, and identical PNG groups kept together. Split seed 0; see evaluation/ocr_dataset/5k-split-20261120/split_manifest.json.

Use a Colab T4: batch 32, max 20 epochs, patience-3 validation-CER early stopping for all models. Up to 125 training steps/epoch, 2,500 over 20 epochs. Start from original public pretrained weights; old fine-tuned checkpoints are not clean baselines for this new test set.

Upload ONLY ocr_dataset-20261120-5k-train-valid.zip, ocr_finetune_code-20261120-earlystop.zip, and ocr_5k_split_review.csv to My Drive/DRIVE. Keep the test ZIP outside this training workflow. It will be used after checkpoint/settings selection is complete.

The review receipt contains only IDs/verdict/reviewer/date from the prior user-confirmed 100-source-crop review. No test text is included in it. It is a sample review, not human verification of every label. Source origin remains metadata; every image/text pair uses the same training/scoring rules.

No models have been trained/evaluated on this new split yet. Preserve previous results as a different experiment. Runtime setup, full batch-32 T4 memory fit and Paddle early-stop/export integration still require Colab validation.


In [ ]:
# Colab: Runtime > Change runtime type > T4 GPU.
# Read-only check of Colab's existing GPU utility; nothing is installed here.
import shutil, subprocess
smi = shutil.which('nvidia-smi')
if smi is None:
    raise RuntimeError('Select T4 GPU in Colab, reconnect, then rerun from the top. No local GPU setup is needed.')
GPU_NAMES = subprocess.check_output(
    [smi, '--query-gpu=name', '--format=csv,noheader'], text=True
).strip().splitlines()
if len(GPU_NAMES) != 1 or not GPU_NAMES[0].strip().endswith('T4'):
    raise RuntimeError(f'This notebook requires one Colab T4 GPU; assigned: {GPU_NAMES}. Select T4 and reconnect.')
print('Using the Colab-provided T4:', GPU_NAMES[0])


In [ ]:
# Check the uploaded bundle before installing packages or downloading weights.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, hashlib
EXPECTED_BUNDLE_SHA256 = '32872fedccdf62823beeecb2766058c2088bd90a3431b4e40ae04974609de027'

def verify_bundle(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Upload the newer bundle to {path}')
    with zipfile.ZipFile(path) as z:
        has_lock = 'weights.lock.json' in z.namelist()
    print('Bundle bytes:', path.stat().st_size, '| weights.lock.json:', has_lock)
    if not has_lock:
        raise RuntimeError('OLD BUNDLE: rename duplicate ocr_dataset-20261120-5k-train-valid.zip files on Drive, upload ocr_dataset-20261120-5k-train-valid.zip and rename it ocr_dataset-20261120-5k-train-valid.zip. Do not continue.')
    with path.open('rb') as f:
        h = hashlib.sha256()
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    if h.hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Bundle SHA-256 differs from the verified local bundle. Stop and check duplicate uploads or ask for an updated notebook.')
    print('Correct bundle verified. Continue to the weights-download cell.')

verify_bundle('/content/drive/MyDrive/DRIVE/ocr_dataset-20261120-5k-train-valid.zip')

# Human review is separate from the automated guards. No prefilled decisions.
import csv
review_path = Path('/content/drive/MyDrive/DRIVE/ocr_5k_split_review.csv')
if not review_path.is_file():
    raise RuntimeError('Upload the completed human_spotcheck.csv as ocr_5k_split_review.csv first.')
review = list(csv.DictReader(review_path.open(encoding='utf-8-sig')))
assert len(review) == 100 and len({r['review_id'] for r in review}) == 100
assert all(r['verdict'].strip().lower() == 'correct' and r['reviewer'].strip() and r['reviewed_at'].strip() for r in review), 'Incomplete/wrong/unreadable sample: inspect and rebuild the candidate rather than bypassing review.'
EXPECTED_IDS = ['SYN2-00035', 'SYN2-00151', 'SYN2-00202', 'SYN2-00353', 'SYN2-00366', 'SYN2-00417', 'SYN2-00465', 'SYN2-00476', 'SYN2-00485', 'SYN2-00502', 'SYN2-00566', 'SYN2-00713', 'SYN2-01034', 'SYN2-01417', 'SYN2-01538', 'SYN2-01639', 'SYN2-01675', 'SYN2-01769', 'SYN2-01932', 'SYN2-01981', 'SYN2-02202', 'SYN2-02683', 'SYN2-02763', 'SYN2-02792', 'SYN2-02835', 'SYN2-03046', 'SYN2-03111', 'SYN2-03269', 'SYN2-03440', 'SYN2-03549', 'SYN2-03585', 'SYN2-03633', 'SYN2-03719', 'SYN2-03749', 'SYN2-03783', 'SYN2-03804', 'SYN2-03819', 'SYN2-03978', 'SYN2-04180', 'SYN2-04364', 'SYN2-04656', 'SYN2-04748', 'SYN2-04857', 'SYN2-04971', 'SYN2-05055', 'SYN2-05201', 'SYN2-05451', 'SYN2-05634', 'SYN2-05664', 'SYN2-05789', 'SYN2-06015', 'SYN2-06071', 'SYN2-06140', 'SYN2-06220', 'SYN2-06246', 'SYN2-06387', 'SYN2-06444', 'SYN2-06445', 'SYN2-06449', 'SYN2-06624', 'SYN2-06627', 'SYN2-06789', 'SYN2-06819', 'SYN2-06916', 'SYN2-07091', 'SYN2-07175', 'SYN2-07192', 'SYN2-07298', 'SYN2-07365', 'SYN2-07531', 'SYN2-07690', 'SYN2-07738', 'SYN2-07869', 'SYN2-07994', 'SYN2-08024', 'SYN2-08118', 'SYN2-08124', 'SYN2-08182', 'SYN2-08206', 'SYN2-08229', 'SYN2-08279', 'SYN2-08319', 'SYN2-08331', 'SYN2-08418', 'SYN2-08536', 'SYN2-08645', 'SYN2-08842', 'SYN2-08871', 'SYN2-08984', 'SYN2-08992', 'SYN2-09059', 'SYN2-09117', 'SYN2-09326', 'SYN2-09467', 'SYN2-09473', 'SYN2-09627', 'SYN2-09651', 'SYN2-09686', 'SYN2-09719', 'SYN2-09953']
assert {r['review_id'] for r in review} == set(EXPECTED_IDS), 'Wrong review sample'


## Download missing initial weights directly into Drive
Run this before the input-check cell. It verifies all nine TrOCR files and English g2 against the uploaded bundle lock. Leave Hugging Face cache folders alone. Hash failure stops this cell; do not train until resolved. Paddle’s training checkpoint is handled later.


In [ ]:
# Run once before the input-check cell below. Downloads go to Drive, not the Mac.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, io, zipfile, urllib.request, importlib.util, subprocess, sys, json
from pathlib import Path
if importlib.util.find_spec('huggingface_hub') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub==0.36.0'], check=True)
from huggingface_hub import hf_hub_download

W = Path('/content/drive/MyDrive/DRIVE/CVDL-OCR-inputs/weights')
def sha(p):
    # Stream the 1.3 GB file instead of reading it all into RAM.
    with Path(p).open('rb') as f:
        h = hashlib.sha256()
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
        return h.hexdigest()

TROCR = {'config.json': '5bda1deab455661feb3d91906656e5600e2ca520d5c00a2a03836614b850c93e', 'generation_config.json': '41149cdcffec4d657f32dfcddd9b208037f01286c9e07945c724908c58ed0193', 'merges.txt': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5', 'model.safetensors': '1cf4a6eedab26afaaf505f1c7f73d9634944924dbd1ed049d569db98039cd596', 'preprocessor_config.json': '2fcc0da9466ee00be0403b26027373039e032820ebac409e207b32e52e52119d', 'special_tokens_map.json': 'c611b1f7d416eb001ee4f293d903ea8c88e703463f1d403f1866a0352743fd00', 'tokenizer_config.json': '5a1356884c6ae736a621841535264ba7c5bebd52f169258add2c48fcbb32d50a', 'vocab.json': '06b4d46c8e752d410213d9548eb27a54db70fda0319b6271fb8d59dead5e1cab'}
PROVENANCE = b'{\n  "repository": "microsoft/trocr-base-printed",\n  "revision": "93450be3f1ed40a930690d951ef3932687cc1892"\n}\n'
PROVENANCE_SHA = 'b6401b3a2237b39a26a9584c5a7aea233cf8183b57bd8cf2c050ef42e6de0a5c'
ENGLISH_G2_SHA = 'e2272681d9d67a04e2dff396b6e95077bc19001f8f6d3593c307b9852e1c29e8'
# Cross-check the constants with the actual uploaded bundle lock, too.
with zipfile.ZipFile('/content/drive/MyDrive/DRIVE/ocr_dataset-20261120-5k-train-valid.zip') as z:
    weight_lock = json.loads(z.read('weights.lock.json'))['models']
assert {**TROCR, 'provenance.json': PROVENANCE_SHA} == weight_lock['trocr']['files_sha256'], 'Bundle TrOCR lock differs; stop'
assert ENGLISH_G2_SHA == weight_lock['easyocr']['files_sha256']['english_g2.pth'], 'Bundle EasyOCR lock differs; stop'

t = W/'trocr'
t.mkdir(parents=True, exist_ok=True)
for name, expected in TROCR.items():
    target = t/name
    needs_download = not target.is_file() or sha(target) != expected
    if needs_download:
        hf_hub_download('microsoft/trocr-base-printed', name,
                        revision='93450be3f1ed40a930690d951ef3932687cc1892',
                        local_dir=str(t), force_download=target.exists())
    assert sha(target) == expected, f'Hash mismatch: {name}. Stop; use the original packaged file instead.'
(t/'provenance.json').write_bytes(PROVENANCE)
assert sha(t/'provenance.json') == PROVENANCE_SHA

e = W/'easyocr'
e.mkdir(parents=True, exist_ok=True)
g2 = e/'english_g2.pth'
if not g2.is_file() or sha(g2) != ENGLISH_G2_SHA:
    with urllib.request.urlopen('https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip', timeout=120) as response:
        data = response.read()
    payload = zipfile.ZipFile(io.BytesIO(data)).read('english_g2.pth')
    assert hashlib.sha256(payload).hexdigest() == ENGLISH_G2_SHA, 'Official English g2 download hash mismatch; stop'
    g2.write_bytes(payload)
assert sha(g2) == ENGLISH_G2_SHA
print('Weights ready and hash-verified in', W)
print('Run the input-check cell below, then setup and CPU dry runs before full training.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, shutil, zipfile, json, time, hashlib
DRIVE_ROOT = Path('/content/drive/MyDrive/DRIVE')  # My Drive > DRIVE, as shown
INPUT = DRIVE_ROOT/'CVDL-OCR-inputs'
OUTPUT = DRIVE_ROOT/'CVDL-OCR-finetune'
BUNDLE = DRIVE_ROOT/'ocr_dataset-20261120-5k-train-valid.zip'
CODE_ZIP = DRIVE_ROOT/'ocr_finetune_code-20261120-earlystop.zip'
WEIGHTS = INPUT/'weights'  # uploaded packaged trocr/ and easyocr/ directories
BATCH_SIZE = 32  #@param [8, 16, 32] {type:'integer'}
assert BATCH_SIZE in (8, 16, 32)
MAX_EPOCHS = 20
RUN_ID = f'ocr5k-split-b{BATCH_SIZE}-e{MAX_EPOCHS}-' + time.strftime('%Y%m%d-%H%M%S',time.gmtime())
WORK = Path('/content/ocr-finetune-5k-split')
WORK.mkdir(exist_ok=True)
def extract_safe(archive,target):
    assert archive.is_file(), f'Missing Drive ZIP: {archive}'
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            assert (target/entry.filename).resolve().is_relative_to(target.resolve()), 'Unsafe ZIP path'
        z.extractall(target)
extract_safe(BUNDLE,WORK/'data')
assert hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest() == 'df561fa3fa1daef863f5fa66acb889c27cffdab3616e141fdd65bcd106914ead', 'Code ZIP hash mismatch'
extract_safe(CODE_ZIP,WORK/'code')
SCRIPTS=WORK/'code/colab/ocr_finetune'
assert (SCRIPTS/'train_trocr.py').is_file(), 'Code ZIP must contain colab/ocr_finetune/'
manifest=json.loads((WORK/'data/bundle_manifest.json').read_text())
print(manifest['counts'])
assert manifest['test_used'] is False and manifest['counts']['test_files']==0
assert (WEIGHTS/'trocr/model.safetensors').is_file(), 'Upload packaged HF trocr weights to Drive'
assert (WEIGHTS/'easyocr/english_g2.pth').is_file(), 'Upload packaged English g2 checkpoint to Drive'
assert OUTPUT.is_relative_to(Path('/content/drive'))

assert manifest["counts"]["train_pairs"] == 4000 and manifest["counts"]["valid_crops"] == 500


In [ ]:
# A GPU must be attached before installing either training environment.
import shutil, subprocess, sys, json
from pathlib import Path
assert len(globals().get('GPU_NAMES', [])) == 1 and GPU_NAMES[0].strip().endswith('T4'), 'Run the T4 runtime check from the top first'
assert shutil.disk_usage('/content').free > 12*1024**3, 'Operational scratch reserve; choose a larger runtime'

# Colab's notebook kernel may use Python 3.13. Training stays on Python 3.12
# in isolated environments, preserving the model-package pins below.
print('Notebook kernel:', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.11'], check=True)
uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv installation completed but its executable is not on PATH; stop and report this error.')
TRAINING_PYTHON = '3.12.12'
subprocess.run([uv, 'python', 'install', TRAINING_PYTHON], check=True)
TORCH_PY = Path('/content/ocr-torch-py312/bin/python')
PADDLE_PY = Path('/content/ocr-paddle-py312/bin/python')
for py in [TORCH_PY, PADDLE_PY]:
    target = py.parent.parent
    if not target.exists():
        subprocess.run([uv, 'venv', '--seed', '--python', TRAINING_PYTHON, str(target)], check=True)
    actual = subprocess.check_output([str(py), '-c', 'import sys; print(".".join(map(str,sys.version_info[:3])))'], text=True).strip()
    assert actual == TRAINING_PYTHON, f'{target} has Python {actual}, expected {TRAINING_PYTHON}; do not reuse this environment.'
    print('Training environment:', py, actual)

def pip(py,*args):
    subprocess.run([str(py), '-m', 'pip', *args], check=True)
pip(TORCH_PY,'install','torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu124')
pip(TORCH_PY,'install','transformers==4.57.1','accelerate==1.11.0','easyocr==1.7.2','numpy==1.26.4','Pillow==11.3.0','PyYAML==6.0.3')
pip(PADDLE_PY,'install','paddlepaddle-gpu==3.2.0','--index-url','https://www.paddlepaddle.org.cn/packages/stable/cu126/')


In [ ]:
def checkout(url,tag,commit,dest):
    dest = Path(dest)
    def git(*args):
        result = subprocess.run(['git', *args], text=True, capture_output=True)
        if result.returncode:
            raise RuntimeError('Git failed: ' + ' '.join(args) + '\n' + result.stdout + result.stderr)
        return result.stdout.strip()
    if not (dest/'.git').exists():
        if dest.exists() and (not dest.is_dir() or any(dest.iterdir())):
            raise RuntimeError(f'{dest} exists but is not a Git checkout. Inspect it; it has not been deleted.')
        git('clone','--depth','1','--branch',tag,url,str(dest))
    actual = git('-C',str(dest),'rev-parse','HEAD')
    origin = git('-C',str(dest),'config','--get','remote.origin.url')
    if origin.rstrip('/') != url.rstrip('/'):
        raise RuntimeError(f'{dest} has a different remote: {origin}. No files were overwritten.')
    if actual != commit:
        raise RuntimeError(f'{dest} is at {actual}, expected {commit}. No checkout/reset was forced.')
    changes = git('-C',str(dest),'status','--porcelain','--untracked-files=no')
    if changes:
        raise RuntimeError(f'{dest} has tracked source changes; inspect them before training:\n{changes}')
    print(f'Ready: {dest} at verified {tag} ({actual[:12]})')
checkout('https://github.com/JaidedAI/EasyOCR.git','v1.7.2','c4f3cd7225efd4f85451bd8b4a7646ae9a092420','/content/EasyOCR')
checkout('https://github.com/PaddlePaddle/PaddleOCR.git','v3.3.2','95dc316a8490a350c1d1fcf3bc80bf83d4e52450','/content/PaddleOCR')
pip(PADDLE_PY,'install','-r','/content/PaddleOCR/requirements.txt')
pip(PADDLE_PY,'install','numpy>=2.0,<2.3','Pillow==11.3.0','PyYAML==6.0.3','setuptools==75.8.0')  # PaddleOCR deps (scipy/opencv) need numpy 2; paddle 3.2 supports it  # paddle imports setuptools; uv's py3.12 venv lacks it
subprocess.run([str(TORCH_PY),'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'],check=True)
subprocess.run([str(PADDLE_PY),'-c','import paddle; assert paddle.is_compiled_with_cuda(); paddle.utils.run_check()'],check=True)


## Official Paddle training checkpoint
The existing inference files cannot train this model. Download the training `.pdparams` directly to Drive; record URL and SHA-256. Never call an untracked existing file an official download.


In [ ]:
import urllib.request
PADDLE_INITIAL=WEIGHTS/'paddle-training/en_PP-OCRv5_mobile_rec_pretrained.pdparams'
PADDLE_INITIAL.parent.mkdir(parents=True,exist_ok=True)
origin=PADDLE_INITIAL.with_suffix('.origin.json')
url='https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams'
if not PADDLE_INITIAL.exists():
    assert not origin.exists(), 'Resolve orphaned provenance record first'
    with urllib.request.urlopen(url) as response, PADDLE_INITIAL.open('xb') as output:
        shutil.copyfileobj(response,output)
    with PADDLE_INITIAL.open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
    origin.write_text(json.dumps(dict(url=url,sha256=digest),indent=2)+'\n')
else:
    assert origin.exists(), 'Existing checkpoint needs its original URL/SHA-256 provenance'
    recorded=json.loads(origin.read_text())
    with PADDLE_INITIAL.open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==recorded['sha256']
    assert recorded['url']==url
print(origin.read_text())


In [ ]:
RUN_ROOT=OUTPUT/'environment'/RUN_ID
RUN_ROOT.mkdir(parents=True,exist_ok=True)
def preserve_text(path, value):
    if path.exists():
        if path.read_text() != value:
            raise RuntimeError(f'Existing run metadata differs: {path}. Preserve this run and choose a new RUN_ID for a new setup.')
    else:
        path.write_text(value)
for py,name in [(TORCH_PY,'torch'),(PADDLE_PY,'paddle')]:
    versions=subprocess.check_output([str(py),'-m','pip','freeze'],text=True)
    preserve_text(RUN_ROOT/(name+'-pip-freeze.txt'), versions)
preserve_text(RUN_ROOT/'input_receipt.json', json.dumps({'bundle_sha256':EXPECTED_BUNDLE_SHA256,'code_sha256':hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest(),'run_id':RUN_ID,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'early_stopping_patience':3,'bundle_manifest_sha256':hashlib.sha256((WORK/'data/bundle_manifest.json').read_bytes()).hexdigest(),'human_review_sha256':hashlib.sha256(review_path.read_bytes()).hexdigest()},indent=2))
preserve_text(RUN_ROOT/'human_spotcheck.csv', review_path.read_text())


## CPU dry runs: 8 pairs, 2 optimiser steps, no saved weights
These are smoke checks, not OCR training results. They write config/logs to Drive. Run all three before starting GPU jobs; inspect any failures. TrOCR can be slow on CPU.


In [ ]:
def launch(py,script,model,initial,dry=False,extra=()):
    output=OUTPUT/model/(RUN_ID+('-dry' if dry else ''))
    if dry and output.exists():  # earlier failed dry run: keep it, but out of the way
        output.rename(output.with_name(output.name+'-failed-'+time.strftime('%H%M%S')))
    args=[str(py),str(SCRIPTS/script),'--data',str(WORK/'data'),'--pretrained',str(initial),
          '--output',str(output),'--seed','0','--batch',str(BATCH_SIZE),'--epochs',str(MAX_EPOCHS),*extra]
    if dry:args.append('--dry-run')
    started=time.perf_counter()
    # Stream output into the notebook (plain subprocess output is not shown in Colab).
    p=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    tail=[]
    for line in p.stdout:
        print(line,end=''); tail=(tail+[line])[-60:]
    if p.wait()!=0:
        raise RuntimeError(f'{model} {"dry run" if dry else "training"} failed (exit {p.returncode}). Last output:\n'+''.join(tail))
    print(model,'dry run' if dry else 'training','seconds:',time.perf_counter()-started,'output:',output)

def smoke_context():
    return {
        'bundle': EXPECTED_BUNDLE_SHA256,
        'code': hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest(),
        'gpu': GPU_NAMES,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'torch_env': subprocess.check_output([str(TORCH_PY), '-m', 'pip', 'freeze'], text=True),
        'paddle_env': subprocess.check_output([str(PADDLE_PY), '-m', 'pip', 'freeze'], text=True),
    }

def smoke_receipt(context):
    digest = hashlib.sha256(json.dumps(context, sort_keys=True).encode()).hexdigest()
    return OUTPUT/'environment'/'smoke-checks'/(digest+'.json')

def require_dry_runs():
    context = smoke_context()
    path = smoke_receipt(context)
    if not path.is_file():
        raise RuntimeError('No successful dry-run receipt for these data/code/environment versions. Run the CPU dry-run cell once in this setup. Do not repeat completed GPU training.')
    record = json.loads(path.read_text())
    if record.get('context') != context or record.get('models') != ['easyocr', 'paddleocr', 'trocr'] or record.get('status') != 'passed':
        raise RuntimeError('Dry-run receipt mismatch; rerun CPU smoke checks for this setup.')
    print('Verified saved successful CPU dry runs:', path)


In [ ]:
# Run all three only when this exact setup has no saved successful receipt.
context = smoke_context()
if smoke_receipt(context).is_file():
    require_dry_runs()
else:
    launch(TORCH_PY,'train_easyocr.py','easyocr',WEIGHTS/'easyocr/english_g2.pth',True,['--repo','/content/EasyOCR'])
    launch(PADDLE_PY,'train_paddle.py','paddleocr',PADDLE_INITIAL,True,['--repo','/content/PaddleOCR'])
    launch(TORCH_PY,'train_trocr.py','trocr',WEIGHTS/'trocr',True)
    
    context = smoke_context()
    path = smoke_receipt(context)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({'status':'passed','context':context,'models':['easyocr','paddleocr','trocr'],'run_id':RUN_ID}, indent=2))
    require_dry_runs()


## Full training on the Colab GPU
Run **one model cell at a time**. Note its exact output folder and wait for a successful finish with `config.json`, `epoch_log.csv` and populated `best/` before starting the next. Inspect the saved model by reloading it before calling it deployable.

All three models now use early stopping: a maximum of 20 epochs, patience 3 validation evaluations without strictly lower crop CER (ties count as no improvement). Every model retains its best validation-CER checkpoint. Paddle catches its validation-plateau signal and exports the saved best checkpoint normally. The new Paddle integration has local tests but still needs its Colab smoke/full-run verification.

Keep the settings unchanged. Send install/loading/CUDA-memory errors instead of changing batch size, learning rate or epochs mid-run. Do not assume an unexpected missing/new-weight warning is harmless.

A disconnect clears session variables and temporary `/content` files. Drive files remain. TrOCR's per-epoch checkpoints are kept on Colab's local disk (only its `best/` goes to Drive), so an interrupted TrOCR run must restart from scratch; EasyOCR and Paddle checkpoints are on Drive. Automatic resume is not implemented: preserve an interrupted folder as incomplete and inspect it before restarting. Do not delete its checkpoints or assume they are all lost. A fresh session creates a new `RUN_ID`.

All 500 validation line images remain in validation CER. No test settings are used. Watch `epoch_log.csv` and the cell output; there is no measured T4 training time yet. Keeping a browser tab open does not guarantee that Colab will keep the runtime connected.


In [ ]:
require_dry_runs()
launch(PADDLE_PY,'train_paddle.py','paddleocr',PADDLE_INITIAL,extra=['--repo','/content/PaddleOCR'])


In [ ]:
require_dry_runs()
launch(TORCH_PY,'train_trocr.py','trocr',WEIGHTS/'trocr')


In [ ]:
require_dry_runs()
launch(TORCH_PY,'train_easyocr.py','easyocr',WEIGHTS/'easyocr/english_g2.pth',extra=['--repo','/content/EasyOCR'])


## After training: separate held-out testing
Return each best/ ZIP, config.json, epoch_log.csv, selection.json, validation_events.jsonl where produced, RUN_ID and input_receipt.json. Select epochs using validation only. Freeze settings/checkpoints before opening ocr_dataset-20261120-5k-test.zip for the 500-example held-out evaluation. That test archive is not mounted/read by this notebook.

Compare original pretrained versus newly fine-tuned models on the same test examples; report CER/WER/exact with the line-image unit and origin breakdown recorded. Earlier fine-tuned models may have trained on this test data, so their historical scores cannot be copied as clean comparisons. The existing original real-photo benchmark can remain a separate supplementary check. No reference matching occurs here; selected new weights and matching JSONL must be generated together later.
